### Embedding models

> https://docs.langchain.com/oss/python/integrations/text_embedding

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector = embeddings.embed_query("hello, world!")
vector[:5]

---

In [4]:
# uv add langchain-community pdfplumber
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [5]:
# uv add langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [6]:
texts[0].page_content

'2026 년 테크노빌드 주식회사(TechnoBuild) 임직원\n통합 가이드북\n문서 번호: TB-HR-2026-001 (Rev 5.0)\n발행일: 2026 년 1 월 15 일\n적용 대상: 테크노빌드 전 임직원 (정규직, 계약직, 파견직 포함)\n주관 부서: 인사문화본부 (HR Culture Division)\n보안 등급: 대외비 (Internal Use Only)\n[목 차]\n1. 회사 개요 (Company Overview)\no 1.1 CEO 인사말\no 1.2 기업 미션 및 비전\no 1.3 핵심 가치 : T-SPIRIT\no 1.4 조직도 및 본부 소개\n2. 인사 및 평가 제도 (HR System)\no 2.1 직급 및 호칭 체계\no 2.2 승진 포인트 제도 (Tech-Point)'

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector = embeddings.embed_documents([text.page_content for text in texts])

In [10]:
vector[0]

[-0.026908852,
 0.016705962,
 0.020847386,
 -0.057225235,
 0.029415336,
 0.0034625577,
 -0.012386412,
 -0.012694033,
 -0.026830126,
 0.0033498423,
 -0.02070041,
 -0.009937428,
 -0.004270815,
 0.035365965,
 0.10984279,
 -0.031752888,
 0.0086158,
 0.01349704,
 -0.013151292,
 -0.010331341,
 0.000559984,
 0.0118949115,
 0.01805831,
 -0.011852158,
 -0.017708883,
 -0.014863579,
 0.01251157,
 0.03955536,
 0.028526608,
 -0.0002604299,
 0.0005274241,
 0.02259332,
 0.0020771418,
 0.020164048,
 -0.005437354,
 0.008755665,
 0.032110576,
 -0.016359655,
 -0.011931744,
 -0.010628185,
 0.005548279,
 0.012156513,
 -0.016741564,
 2.3553319e-05,
 -0.0010883037,
 -0.026042053,
 -0.021717627,
 -0.016530583,
 -0.0039236876,
 0.015560446,
 0.0034956574,
 -0.0077106296,
 -0.01931446,
 -0.17190185,
 -0.0010324584,
 -0.002993755,
 -0.028370203,
 0.0064161704,
 -0.021233235,
 -0.008801993,
 -0.042520713,
 0.03490782,
 -0.025517154,
 -0.027432408,
 -0.0004582444,
 -0.02032746,
 0.029202983,
 0.027205855,
 0.00324

In [11]:
embeddings.model

'models/gemini-embedding-001'

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    store,
    namespace=embeddings.model
)

In [15]:
from langchain_core.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(
    texts,
    cached_embedder
)

In [16]:
query = "임직원 통합 가이드북의 발행일은?"

In [18]:
results = vectorstore.similarity_search(query, k=1)
results

[Document(id='674c5d2a-5437-4790-83c2-48f002a2e27e', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 0, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='2026 년 테크노빌드 주식회사(TechnoBuild) 임직원\n통합 가이드북\n문서 번호: TB-HR-2026-001 (Rev 5.0)\n발행일: 2026 년 1 월 15 일\n적용 대상: 테크노빌드 전 임직원 (정규직, 계약직, 파견직 포함)\n주관 부서: 인사문화본부 (HR Culture Division)\n보안 등급: 대외비 (Internal Use Only)\n[목 차]\n1. 회사 개요 (Company Overview)\no 1.1 CEO 인사말\no 1.2 기업 미션 및 비전\no 1.3 핵심 가치 : T-SPIRIT\no 1.4 조직도 및 본부 소개\n2. 인사 및 평가 제도 (HR System)\no 2.1 직급 및 호칭 체계\no 2.2 승진 포인트 제도 (Tech-Point)')]